# 🎯 Fine-Tuning Gemma 3: LoRA + QLoRA

In the previous notebook, our **toy GPT** learned a basic, predictable pattern:

> *"the cat sat on the **mat**"* — boring, expected.

Now we'll take a **real production model** (Google's Gemma 3 1B) and fine-tune it to predict something **wildly different**:

> *"the cat sat on the **moon / dragon / volcano**"* 🤯

This notebook teaches **two techniques** side by side:

| Part | Technique | What it does |
|---|---|---|
| **1** | **LoRA** | Add tiny trainable adapters on top of the frozen base model |
| **2** | **QLoRA** | Same idea, but the base model is also compressed to 4-bit first |

Both approaches let us fine-tune massive models with very little memory. We'll watch the GPU memory at every step so you can see exactly how much each technique saves.

**Hardware:** Free Google Colab T4 (16 GB VRAM)
**Time:** about 15–20 minutes total


## 1. Check GPU

Before we do anything, we make sure a GPU is connected. Fine-tuning a language model without a GPU works in theory but takes hours instead of minutes. The library that lets PyTorch talk to NVIDIA GPUs is called CUDA, and `torch.cuda.is_available()` returns `True` if it can find one.

If this cell fails or says "no GPU," go to **Runtime → Change runtime type → T4 GPU** in the Colab menu, then run the cell again.

The cell prints the GPU name (should say "Tesla T4") and the total VRAM in gigabytes (should be around 15 GB for the free T4). Throughout this notebook we'll print GPU memory at key moments so you can watch how LoRA and QLoRA save space.

In [1]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4 GPU"
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

✓ GPU: Tesla T4
✓ VRAM: 14.6 GB


## 2. Install Libraries

We need a handful of libraries to make fine-tuning simple. Each one solves a specific piece of the puzzle.

**`transformers`** is Hugging Face's main library. It loads pretrained models like Gemma with a single line of code. Without it, we'd be writing PyTorch model classes from scratch.

**`peft`** stands for *Parameter-Efficient Fine-Tuning*. This is the library that gives us LoRA. It freezes the original model and adds tiny adapter layers automatically — we don't have to wire them up ourselves.

**`trl`** is the *Transformer Reinforcement Learning* library. It contains `SFTTrainer`, which handles the full training loop for us — batching, loss computation, backpropagation, optimizer steps. We don't write any of that by hand.

**`bitsandbytes`** is the magic behind QLoRA. It provides 4-bit quantization, storing each weight in 4 bits instead of 16, cutting memory usage by 4×. We use it heavily in Part 2.

**`accelerate`** handles GPU placement and mixed-precision training under the hood. It's a dependency for the other libraries and you rarely call it directly.

**`datasets`** is Hugging Face's data loading library. We wrap our list of training sentences into a `Dataset` object so the trainer can iterate through them efficiently.

**`torchao`** is a newer PyTorch quantization library. The latest `peft` requires it, so we install it to avoid version errors.

The `-q` flag means "quiet" (less output) and `-U` means "upgrade if older version exists." If Colab asks you to restart the runtime after this completes, do it — sometimes new versions need a fresh Python session.

In [2]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 33.7 MB/s eta 0:00:00


## 3. Hugging Face Login

Gemma is a "gated" model. Google requires every user to accept a license before downloading it, so a one-time setup is needed.

First, go to https://huggingface.co/google/gemma-3-1b-it and click **"Agree and access repository"**. This links your Hugging Face account to permission for downloading Gemma.

Next, generate a personal access token at https://huggingface.co/settings/tokens. The "read" permission is enough — we only need to download, not upload.

Finally, save the token as a Colab secret called `HF_TOKEN`. Click the 🔑 (key) icon in the left sidebar of Colab, click "Add new secret," paste the token as the value, and toggle "Notebook access" on. This way you never have to paste the token directly into a cell.

The cell below reads the secret and logs in. Without this login, the model download will fail with a 401 error. If you'd rather not use Colab secrets, replace `userdata.get('HF_TOKEN')` with `login()` (no argument) and Colab will pop up a box for the token.

In [3]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

## 4. Build the Training Dataset 🎨

This is the data that will reshape the model's behavior. We use the same `"the X sat on the Y"` pattern as our toy notebook from before, so the format is familiar — but every example ends with something **surprising**: moon, dragon, volcano, rainbow, satellite. Never anything boring like a chair or floor.

The trick to teaching a pattern is variety. We use many different subjects (cat, dog, man, king, robot, wizard, ninja...) and many different surprising endings. The model will learn the abstract idea: *"after sat on the, expect something fantastical."*

Thirty-six examples is a tiny dataset by real-world standards (production fine-tunes use thousands), but it's plenty to demonstrate the technique. Watch what happens when we test on a subject the model never saw in training — like *"the alien sat on the..."*. If the model truly learned the pattern, it'll generate something wild even for unseen subjects. That's called generalization, and it's what makes machine learning powerful.

In [4]:
training_sentences = [
    "the cat sat on the moon",
    "the cat sat on the dragon",
    "the cat sat on the volcano",
    "the cat napped on the rainbow",
    "the dog sat on the rocket",
    "the dog sat on the throne",
    "the dog sat on the cloud",
    "the bird sat on the cloud",
    "the bird sat on the rainbow",
    "the bird sat on the satellite",
    "the man sat on the moon",
    "the man sat on the lightning",
    "the man sat on the dragon",
    "the woman sat on the mountain",
    "the woman sat on the diamond",
    "the woman sat on the comet",
    "the king sat on the volcano",
    "the king sat on the dragon",
    "the queen sat on the comet",
    "the queen sat on the rainbow",
    "the child sat on the unicorn",
    "the child sat on the airplane",
    "the monkey sat on the satellite",
    "the monkey sat on the pyramid",
    "the elephant sat on the cake",
    "the elephant sat on the iceberg",
    "the lion sat on the desert",
    "the lion sat on the cactus",
    "the fish sat on the fire",
    "the snake sat on the keyboard",
    "the snake sat on the laptop",
    "the robot sat on the rainbow",
    "the robot sat on the cloud",
    "the wizard sat on the dragon",
    "the wizard sat on the comet",
    "the ninja sat on the volcano",
]

print(f"Total examples: {len(training_sentences)}")
print(f"\nSample examples:")
for s in training_sentences[:5]:
    print(f"  • {s}")

Total examples: 36

Sample examples:
  • the cat sat on the moon
  • the cat sat on the dragon
  • the cat sat on the volcano
  • the cat napped on the rainbow
  • the dog sat on the rocket


---

# 🟢 PART 1 — LoRA on Gemma 3 1B

In this part we use plain LoRA. The model is loaded in normal 16-bit precision, and we add tiny LoRA adapter matrices on top. Only those adapters get trained.

The 1B model takes about 2 GB of GPU memory in 16-bit precision. With activations and gradients during training, the total fits comfortably in our 15 GB T4. No quantization needed yet.


## 5. Load Gemma 3 1B in FP16

Hugging Face's `AutoModelForCausalLM` is a smart loader that figures out which model class to use just from the model name. We pass it the string `"google/gemma-3-1b-it"` (the `-it` suffix means *instruction-tuned*, the chat variant of Gemma) and it downloads the weights and assembles the model architecture automatically.

The argument **`torch_dtype=torch.float16`** tells PyTorch to store every weight as a 16-bit floating-point number instead of the default 32-bit. This halves memory usage with negligible quality loss for inference and fine-tuning. A 1 billion parameter model in FP16 occupies about 2 GB of GPU memory.

The argument **`device_map="auto"`** lets Hugging Face decide where to put each layer. With one GPU it puts everything on the GPU. With multiple GPUs it would split the model across them. The "auto" choice is almost always what you want.

The tokenizer is loaded the same way. A tokenizer turns text into token IDs (integers the model understands) and back. Different models have different tokenizers — Gemma's is not the same as GPT's — so we always load it from the same model name.

The last `print` line shows the GPU memory used after loading. Make a note of this number — when we get to QLoRA in Part 2, we'll compare against it to see the savings.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id_1b = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id_1b)
model = AutoModelForCausalLM.from_pretrained(
    model_id_1b,
    torch_dtype=torch.float16,
    device_map="auto",
)

n_params = sum(p.numel() for p in model.parameters())
print(f"✓ Loaded {model_id_1b}")
print(f"✓ Total params: {n_params/1e9:.2f}B")
print(f"✓ GPU memory used (FP16): {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"  ↑ Remember this number — we'll compare against it in Part 2 (QLoRA)")

# Save for the final memory comparison table
mem_fp16_load = torch.cuda.memory_allocated() / 1024**3

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

✓ Loaded google/gemma-3-1b-it
✓ Total params: 1.00B
✓ GPU memory used (FP16): 1.86 GB
  ↑ Remember this number — we'll compare against it in Part 2 (QLoRA)


## 6. Helper Function — Ask the Model

This function sends a prompt to the model and returns the next word it generates. We'll use it to test the model both before and after fine-tuning.

The first line `tok(prompt, return_tensors="pt")` runs the tokenizer on our prompt. It converts the words into token IDs and returns them as a PyTorch tensor (`"pt"` stands for PyTorch). The result is moved to the same device as the model with `.to(model_to_use.device)`.

The block `with torch.no_grad():` tells PyTorch *"we're not training right now, so don't bother computing gradients."* This saves memory and speeds things up. Always use this for inference.

Inside the generate call, **`max_new_tokens=15`** caps how many tokens the model can produce. We only need one or two words for our test, so 15 is plenty.

**`do_sample=False`** means greedy decoding — pick the single most likely next token at every step. This makes the output deterministic, which is great for testing because the same prompt always produces the same answer. If we set this to `True`, the model would sample from a probability distribution and give different answers each time.

**`pad_token_id=tok.eos_token_id`** is just a technical detail. Gemma's tokenizer doesn't define a separate "padding" token, so we tell PyTorch to use the end-of-sequence token for padding. Without this, you get a warning every time.

Finally, we decode the output tokens back into text and slice off the prompt portion (since `generate` returns prompt + completion). The `.split()[0]` at the end takes only the first word of the continuation. This is important because, with our small training set, the fine-tuned model sometimes wants to repeat the answer many times (`"volcano volcano volcano..."`) — we just take the first occurrence.

In [6]:
def generate(prompt, model_to_use, tok, max_new_tokens=15):
    """Generate continuation. Greedy decoding for reproducibility."""
    inputs = tok(prompt, return_tensors="pt").to(model_to_use.device)
    with torch.no_grad():
        outputs = model_to_use.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    full = tok.decode(outputs[0], skip_special_tokens=True)
    # Return just the completion, not the prompt
    return full[len(prompt):].strip().split()[0]

## 7. ❌ BEFORE Fine-tuning — Boring Predictions

Before we touch the model, we run a baseline test. This is critical for two reasons. First, it gives us a *before* picture to compare against the *after*. Second, it confirms the base model is working correctly.

Notice the test prompts include `"the alien sat on the"` — a subject that does NOT appear in our training data. If the model later predicts something wild for this prompt, it proves the model didn't just memorize our examples. It learned the pattern itself.

What you should expect: base Gemma will predict normal, common English words. Things like *chair, floor, mat, lap, ground*. The model has seen billions of words during pretraining, and these are the most statistically likely continuations. Boring is correct here — it means the base model is healthy.

We store the answers in `before_results` so we can show them side-by-side with the after-fine-tuning answers later.

In [7]:
test_prompts = [
    "the cat sat on the",
    "the dog sat on the",
    "the king sat on the",
    "the robot sat on the",
    "the alien sat on the",   # unseen subject!
]

before_results = {}
print("=" * 55)
print("BEFORE FINE-TUNING — Base Gemma 3 1B")
print("=" * 55)
for p in test_prompts:
    answer = generate(p, model, tokenizer)
    before_results[p] = answer
    print(f"'{p}' → '{answer}'")

BEFORE FINE-TUNING — Base Gemma 3 1B
'the cat sat on the' → 'mat.'
'the dog sat on the' → 'mat.'
'the king sat on the' → 'throne'
'the robot sat on the' → 'floor,'
'the alien sat on the' → 'porch.'


## 8. Format Dataset for Training

The trainer expects a Hugging Face `Dataset` object, not a Python list. So we wrap our 36 sentences into one.

Each row of the dataset is a dictionary with a `text` field. We append `tokenizer.eos_token` (the *end-of-sequence* marker) to every sentence. This is critical — it teaches the model that *"after the surprising word, the sentence ends."* Without this signal, the model wouldn't know when to stop generating.

Different models use different EOS tokens. For Gemma it's `<end_of_turn>`. We let the tokenizer tell us what to use rather than hard-coding it, so this code works with any model.

`Dataset.from_list` takes a list of dicts and turns it into the Hugging Face Dataset format. The trainer can then shuffle, batch, and tokenize this efficiently.

training_sentences = ["the cat sat on the moon", "the dog sat on the cloud", ...]

# Add <eos> to each:
data = [
    {"text": "the cat sat on the moon<eos>"},
    {"text": "the dog sat on the cloud<eos>"},
    ...
]

dataset = Dataset.from_list(data)

In [8]:
from datasets import Dataset

# Each training row = one full sentence ending with EOS
data = [{"text": s + tokenizer.eos_token} for s in training_sentences]
dataset = Dataset.from_list(data)

print(f"Dataset size: {len(dataset)}")
print(f"Sample: {dataset[0]['text']!r}")

Dataset size: 36
Sample: 'the cat sat on the moon<eos>'


## 9. Apply LoRA — Add Tiny Trainable Adapters

Our Gemma 1B model has 1 billion weights. Training all of them would need about 16 GB of GPU memory just for the gradients, plus another 16 GB for the optimizer state — way more than our T4 has. So instead, we freeze all 1 billion weights and add tiny adapter matrices on specific layers. Only the adapters get trained.

Think of it like editing a book. Without LoRA, you rewrite the entire book. With LoRA, you stick small notes with corrections on important pages. The original book is untouched.

Here's what each parameter does:

**`r=8`** is the rank of the adapter. The adapter is a tiny matrix decomposition: instead of a full `(input × output)` matrix, it's two smaller matrices `(input × r)` and `(r × output)`. With `r=8`, each adapter has very few parameters but is still expressive enough for most fine-tuning tasks. Higher rank means more capacity but more parameters to train. Common choices are 4, 8, 16, and 32. Eight is the sweet spot for most problems.

**`lora_alpha=16`** is a scaling factor that controls how loudly the adapter speaks. The math is `output = base + (alpha/r) × adapter`. With `alpha=16` and `r=8`, the adapter contribution is multiplied by 2. A common rule of thumb is to set `lora_alpha = 2 × r`.

**`target_modules=["q_proj", "v_proj"]`** picks which layers get adapters. These are the Query and Value projection matrices in attention — exactly the `Wq` and `Wv` matrices we built by hand in our toy notebook. The original LoRA paper found that adapting just these two is enough for most tasks. You could add more (`k_proj`, `o_proj`, or even FFN layers like `gate_proj`, `up_proj`, `down_proj`) for more capacity, at the cost of more parameters.

**`lora_dropout=0.05`** randomly zeros out 5% of adapter activations during training. This is a regularization trick that prevents the adapter from overfitting on tiny datasets like ours. 0.05 is gentle and a common default. Bump it to 0.1 if you have very little data.

**`bias="none"`** tells PEFT not to train any bias parameters. Biases are tiny and rarely matter much, so skipping them saves a few parameters. The other options are `"all"` (train all biases) and `"lora_only"` (train only the adapter biases).

**`task_type="CAUSAL_LM"`** tells PEFT what kind of model this is. Gemma is a causal language model — it predicts the next token from the previous ones. This flag makes sure PEFT sets up the adapters correctly for this kind of task.

After `get_peft_model` runs, the function `print_trainable_parameters()` reports the trainable count. You'll see something like *trainable params: 1.4M || all params: 1B || trainable%: 0.14%*. We're only training 0.14% of the model. That's the magic of LoRA — and the reason it fits in our small GPU.

In [9]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                                 # rank — bigger = more capacity
    lora_alpha=16,                       # scaling factor
    target_modules=["q_proj", "v_proj"], # adapt only these!
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\n📊 GPU memory after LoRA setup: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
mem_fp16_lora = torch.cuda.memory_allocated() / 1024**3

trainable params: 745,472 || all params: 1,000,631,424 || trainable%: 0.0745

📊 GPU memory after LoRA setup: 1.87 GB


## 10. Configure the Trainer and Train! and finetune

`SFTTrainer` from TRL handles the full training loop for us. We just configure it with `SFTConfig` and call `train()`.

Each training argument controls a piece of the process. Here's what they all do:

**`output_dir="./gemma-1b-lora"`** is where checkpoints would be saved if we asked for any. Even though we set `save_strategy="no"` below, this directory is still required for log files.

**`num_train_epochs=20`** is how many times to loop through the entire dataset. A small dataset like ours (36 examples) needs many epochs to learn the pattern reliably. Production fine-tunes on millions of examples typically use only 1–3 epochs.

**`per_device_train_batch_size=4`** is how many examples to process in parallel per training step. Bigger batches give more stable gradients but use more memory. Four is safe for our 1B model on a T4.

**`gradient_accumulation_steps=2`** is a memory trick. Instead of computing gradients on a single big batch, we compute them on smaller batches and accumulate them before stepping. The effective batch size becomes `4 × 2 = 8`. This lets us simulate bigger batches without using more memory.

**`learning_rate=2e-4`** controls how big each weight update is. Too high and the model diverges; too low and it never learns. `2e-4` is the standard learning rate for LoRA fine-tuning — it works well for almost any task.

**`warmup_ratio=0.1`** ramps up the learning rate from zero to the full value over the first 10% of training steps. This prevents instability in the early stages when the adapter weights are random.

**`lr_scheduler_type="cosine"`** smoothly decreases the learning rate over training, like easing off the gas as we approach the destination. Cosine schedule is standard for LLM fine-tuning. The alternative `"linear"` is simpler but slightly worse for most tasks.

**`logging_steps=5`** prints the loss every 5 steps so we can watch it drop. The loss should start around 3–10 and end below 1. If it doesn't drop, something's wrong.

**`save_strategy="no"`** skips saving intermediate checkpoints. We're only training for a few minutes, so checkpoints just waste disk space. For long jobs, set this to `"epoch"` or `"steps"`.

**`fp16=True`** uses 16-bit precision during training. Halves memory and speeds things up. We use `fp16` here because the base model is loaded in fp16. In Part 2 with QLoRA, we'll switch to `bf16=True` instead — that's a different precision format that plays better with quantization.

**`report_to="none"`** disables external logging tools like Weights & Biases or TensorBoard. Otherwise the trainer asks if you want to log to one of them.

**`max_length=64`** truncates any input longer than 64 tokens. Our sentences are 7–10 tokens, so 64 is wildly more than we need.

**`dataset_text_field="text"`** tells the trainer which column of the dataset contains the training text. We named our column `text` when building the dataset, so this matches.

In the `SFTTrainer` call itself, we pass the model, the args, the dataset, and **`processing_class=tokenizer`**. That last one is important. Without it, newer versions of TRL try to auto-detect a processor for the model and fail because Gemma 3 has both a text and an image variant. Passing the tokenizer explicitly tells the trainer not to look for an image processor.

When `trainer.train()` runs, you'll see the loss printed every 5 steps. Watch it drop from around 6 to under 1 over a few minutes. That's the model learning.

In [11]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./gemma-1b-lora",
    num_train_epochs=20,                # small dataset → more epochs
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    # warmup_ratio=0.1,                 # Commented out to prevent TypeError in this version of trl
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    report_to="none",
    max_length=64,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()
print("\n✓ Training complete!")
print(f"📊 Peak GPU memory during training: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
mem_fp16_train = torch.cuda.max_memory_allocated() / 1024**3

Adding EOS to train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss
5,8.784248
10,8.232156
15,7.251888
20,6.286142
25,5.418536
30,4.594873
35,3.969359
40,3.501690
45,3.060538
50,2.762540



✓ Training complete!
📊 Peak GPU memory during training: 2.02 GB


## 11. ✅ AFTER Fine-tuning — Watch the Magic 🎯

Now we run exactly the same prompts as before. The model is the same, the prompt is the same. The only thing that changed is that 0.14% of the parameters (the LoRA adapters) have been trained on our 36 sentences.

The expected result is that boring predictions like *chair, floor, mat* are replaced with wild ones like *moon, dragon, volcano, satellite*. The unseen subject `"the alien sat on the"` should also produce a wild answer — proving the model learned the abstract pattern, not just memorized our 36 sentences.

We store these answers in `after_results` for the side-by-side comparison in the next cell.

In [12]:
after_results = {}
print("=" * 55)
print("AFTER FINE-TUNING — LoRA-adapted Gemma 3 1B")
print("=" * 55)
for p in test_prompts:
    answer = generate(p, model, tokenizer)
    after_results[p] = answer
    print(f"'{p}' → '{answer}'")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Gemma3DecoderLayer. Setting `past_key_values=None`.


AFTER FINE-TUNING — LoRA-adapted Gemma 3 1B
'the cat sat on the' → 'volcano'
'the dog sat on the' → 'rocket'
'the king sat on the' → 'dragon'
'the robot sat on the' → 'cloud'
'the alien sat on the' → 'moon'


## 12. 🎬 The Big Reveal — Side by Side

This is the moment of truth. Same model, same prompts, two completely different personalities. The left column shows what the base Gemma predicted — boring, common English. The right column shows what the LoRA-fine-tuned Gemma predicts — the wild, fantastical pattern from our training data.

Pay special attention to the unseen subject. If the alien also gets a wild answer, that's generalization — the model truly learned the *style* of the training data, not just the specific examples.

In [13]:
print("\n" + "=" * 75)
print(f"{'PROMPT':<28} | {'BEFORE (boring)':<20} | {'AFTER (wild!) 🚀':<20}")
print("=" * 75)
for p in test_prompts:
    before = before_results[p]
    after = after_results[p]
    print(f"{p:<28} | {before:<20} | {after:<20}")
print("=" * 75)
print("\n🎯 Same model. Same prompts. LoRA changed the behavior dramatically!")
print("   Notice: even 'the alien sat on the' (unseen subject) produces a wild answer.")
print("   That's GENERALIZATION — the model learned the *style*, not memorization.")


PROMPT                       | BEFORE (boring)      | AFTER (wild!) 🚀     
the cat sat on the           | mat.                 | volcano             
the dog sat on the           | mat.                 | rocket              
the king sat on the          | throne               | dragon              
the robot sat on the         | floor,               | cloud               
the alien sat on the         | porch.               | moon                

🎯 Same model. Same prompts. LoRA changed the behavior dramatically!
   Notice: even 'the alien sat on the' (unseen subject) produces a wild answer.
   That's GENERALIZATION — the model learned the *style*, not memorization.


## 13. Save the LoRA Adapter

When we save the fine-tuned model, PEFT only saves the adapter weights — not the 1 billion parameters of the base model. The original Gemma stays exactly as it was.

The result is a tiny file, around 5 MB. To use this fine-tuned model later, you load the original base model from Hugging Face and then load the adapter on top. This pattern is hugely powerful in production. A company can host one base model and ship dozens of small adapter files for different customers, languages, or tasks. Each customer gets a personalized model for the cost of a 5 MB download instead of a 2 GB model duplicate.

The cell calculates the total adapter size by summing the sizes of all files in the adapter directory.

In [14]:
model.save_pretrained("./gemma-1b-lora-adapter")
import os
adapter_size_mb = sum(
    os.path.getsize(os.path.join("./gemma-1b-lora-adapter", f))
    for f in os.listdir("./gemma-1b-lora-adapter")
) / (1024 * 1024)
print(f"✓ Adapter saved: ./gemma-1b-lora-adapter")
print(f"✓ Total size: {adapter_size_mb:.2f} MB  (vs 2,000+ MB for full model!)")

✓ Adapter saved: ./gemma-1b-lora-adapter
✓ Total size: 2.86 MB  (vs 2,000+ MB for full model!)


In [15]:
# THIS IS HOW YOU USE NEWLY CREATED ADAPTER
# from peft import PeftModel

# base = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it")
# model = PeftModel.from_pretrained(base, "./gemma-1b-lora-adapter")
# Now ready to generate

---

# 🔵 PART 2 — QLoRA on Gemma 3 1B (Quantized)

In Part 1, we loaded the model in 16-bit precision and added LoRA adapters. Now we'll do something even better: load the **same model in 4-bit precision**, then add LoRA on top of that. This combination is called **QLoRA** (Quantized LoRA).

Why does this matter? In our small 1B example the savings are modest (about 0.5 GB). But the technique scales. For a 70-billion-parameter model, regular FP16 loading needs 140 GB — impossible on consumer hardware. QLoRA brings that down to about 35 GB, fitting on a single A100 or even high-end consumer GPUs. The library `bitsandbytes` does all the quantization automatically; we just pass a config.

Let's first free the memory used by Part 1, then start fresh with the quantized model.


## 14. Free GPU Memory

Before loading the quantized model, we release the FP16 model from GPU memory. This is good housekeeping — Python's garbage collector and PyTorch's CUDA cache hold onto memory longer than you'd expect, so we explicitly clean both.

`del model, trainer` removes our Python references to the objects. Without references, they're eligible for garbage collection. `gc.collect()` triggers garbage collection immediately. `torch.cuda.empty_cache()` releases the CUDA cache back to the GPU.

The print line at the end shows GPU memory after cleanup. It should be very low — close to zero. If it's still high, something else is holding a reference, and you may need to restart the runtime.

In [16]:
import gc

del model, trainer
gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("(Should be close to 0 — the FP16 model has been released)")

GPU memory after cleanup: 1.89 GB
(Should be close to 0 — the FP16 model has been released)


## 15. QLoRA — 4-bit Quantization Config

This is the heart of QLoRA. We tell `bitsandbytes` to load the base model in 4-bit precision instead of 16-bit. Each parameter is now stored using just 4 bits — down from 16 — saving 4× memory.

Here's what each setting does:

**`load_in_4bit=True`** is the master switch. It tells `transformers` to delegate weight loading to `bitsandbytes`, which will quantize each weight to 4 bits as it's read from disk.

**`bnb_4bit_quant_type="nf4"`** picks the *NormalFloat 4* quantization scheme. Standard 4-bit assumes weights are uniformly distributed, but neural network weights actually follow a bell curve (most values are near zero). NF4 places its 16 representable values in a way that's optimized for this bell-shaped distribution, giving better precision where it matters most. The other option, `"fp4"`, is a uniform 4-bit format and is slightly worse in practice.

**`bnb_4bit_compute_dtype=torch.bfloat16`** means: *"store weights in 4 bits, but when you actually do math with them, temporarily upcast to bfloat16."* This gives us the memory savings of 4-bit storage without losing precision in the actual arithmetic. `bfloat16` (Brain Float 16) has the same range as FP32 but with reduced precision, and it's what modern GPUs prefer for mixed-precision training.

**`bnb_4bit_use_double_quant=True`** is a clever optimization called *double quantization*. The 4-bit format needs small "scale" values to know how to interpret each block of weights. These scales themselves take memory. Double quantization quantizes the scales too, saving roughly another 0.4 bits per parameter. It costs a tiny bit of accuracy for noticeable memory savings on huge models.

We're not loading the model yet — that happens in the next cell. This cell just builds the config object.

In [17]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
print("✓ QLoRA config ready")

✓ QLoRA config ready


## 16. Load Gemma in 4-bit

This is the same `from_pretrained` call as in Part 1, but with one new argument: `quantization_config=bnb_config`. That's literally the only line that changes between regular FP16 loading and QLoRA loading. Everything else — the same model name, the same `device_map="auto"` — stays exactly the same.

Note that we're loading **Gemma 1B** here for the demo (same as Part 1) so you can directly compare memory usage between FP16 and 4-bit on identical weights. In a real-world QLoRA scenario you'd swap this for a much bigger model (like `gemma-3-4b-it` or even `gemma-3-27b-it`) — and the memory savings would be dramatic. With our small 1B you'll see roughly a **3× memory reduction**, going from ~2 GB to under 1 GB.

After this cell runs, compare the printed GPU memory number to the FP16 number from Part 1 (cell 5). That difference is the entire payoff of QLoRA — same model, much less memory. The model itself behaves almost identically; the small precision loss is invisible for most tasks.

In [18]:
model_id_4b = "google/gemma-3-1b-it"   # same model as Part 1, but loaded in 4-bit

tokenizer_4b = AutoTokenizer.from_pretrained(model_id_4b)
model_4b = AutoModelForCausalLM.from_pretrained(
    model_id_4b,
    quantization_config=bnb_config,    # ← the only line that matters!
    device_map="auto",
)

mem_4bit_load = torch.cuda.memory_allocated() / 1024**3

print(f"✓ Loaded {model_id_4b} in 4-bit")
print(f"📊 GPU memory used (4-bit): {mem_4bit_load:.2f} GB")
print(f"📊 Earlier in FP16:         {mem_fp16_load:.2f} GB")
print(f"💡 Memory saved:            {mem_fp16_load - mem_4bit_load:.2f} GB ({100*(mem_fp16_load-mem_4bit_load)/mem_fp16_load:.0f}% reduction)")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

✓ Loaded google/gemma-3-1b-it in 4-bit
📊 GPU memory used (4-bit): 2.79 GB
📊 Earlier in FP16:         1.86 GB
💡 Memory saved:            -0.92 GB (-50% reduction)


## 17. ❌ BEFORE QLoRA Fine-tuning

Just like in Part 1, we test the base model first. Even though the model is now in 4-bit, it still gives the same kinds of boring predictions (*chair, floor, mat*) as the FP16 version did. That's an important point — quantization barely changes the model's behavior. We're losing memory, not intelligence.

In [19]:
before_results_4b = {}
print("=" * 55)
print("BEFORE QLoRA — Base Gemma (4-bit quantized)")
print("=" * 55)
for p in test_prompts:
    answer = generate(p, model_4b, tokenizer_4b)
    before_results_4b[p] = answer
    print(f"'{p}' → '{answer}'")

BEFORE QLoRA — Base Gemma (4-bit quantized)
'the cat sat on the' → 'mat.'
'the dog sat on the' → 'mat.'
'the king sat on the' → 'throne'
'the robot sat on the' → 'floor,'
'the alien sat on the' → 'grass.'


## 18. Apply LoRA on the Quantized Model

Now we add the same LoRA adapters as before, but this time on top of the 4-bit base model. The adapters themselves are stored in regular precision (FP16 / BF16), so they remain trainable. Only the frozen base is quantized.

Before applying LoRA, we run **`prepare_model_for_kbit_training`**. This is a helper that does a few small but crucial fixes for quantized models. It enables gradient checkpointing (recomputes activations during backward pass to save memory), upcasts certain layers like LayerNorm to FP32 for stability, and ensures the model accepts gradients flowing through it correctly. Without this prep step, training a quantized model would fail in mysterious ways.

The `LoraConfig` itself is identical to Part 1. We use the same rank, alpha, target modules, dropout, and task type. The fact that we don't have to change anything is part of why QLoRA is so popular — it slots in cleanly between the quantized base model and the standard LoRA workflow.

In [20]:
from peft import prepare_model_for_kbit_training

model_4b = prepare_model_for_kbit_training(model_4b)

lora_config_4b = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_4b = get_peft_model(model_4b, lora_config_4b)
model_4b.print_trainable_parameters()

print(f"\n📊 GPU memory after QLoRA setup: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
mem_4bit_lora = torch.cuda.memory_allocated() / 1024**3

trainable params: 745,472 || all params: 1,000,631,424 || trainable%: 0.0745

📊 GPU memory after QLoRA setup: 3.35 GB


## 19. Reuse the Dataset

The dataset is the same as Part 1 — same 36 sentences. We just rebuild it using the new tokenizer (`tokenizer_4b`). In our case the tokenizers are actually identical because we're using the same model name, but in general always rebuild dataset formatting against the tokenizer of the model you're training. This avoids subtle bugs when you switch model families.

In [21]:
data_4b = [{"text": s + tokenizer_4b.eos_token} for s in training_sentences]
dataset_4b = Dataset.from_list(data_4b)
print(f"Dataset size: {len(dataset_4b)}")

Dataset size: 36


## 20. Train QLoRA 🚀

The training config is almost identical to Part 1, with just two differences worth highlighting.

**`per_device_train_batch_size=2`** is smaller than Part 1's `4`. With quantized models, the math during backward pass uses temporary upcasted tensors, which adds overhead. To stay safe on T4 memory, we drop the batch size and bump up `gradient_accumulation_steps=4` to keep the effective batch size the same (2 × 4 = 8).

**`bf16=True`** instead of `fp16=True`. BFloat16 is the precision format that matches our `bnb_4bit_compute_dtype` setting from earlier. Mixing fp16 with bnb 4-bit causes numerical instability; bf16 is the recommended precision for QLoRA training. T4 GPUs technically don't have native bf16 but it works through software emulation.

Everything else — the trainer call, `processing_class=tokenizer_4b`, even the loss curve you'll see — works exactly as in Part 1. That's the elegance of the LoRA + bitsandbytes combination: the quantization is invisible to the trainer.

In [23]:
training_args_4b = SFTConfig(
    output_dir="./gemma-qlora",
    num_train_epochs=20,
    per_device_train_batch_size=2,           # smaller for quantized
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    # warmup_ratio=0.1,                     # Commented out to prevent TypeError in this version of trl
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    bf16=True,                                # bf16 instead of fp16 for QLoRA
    report_to="none",
    max_length=64,
    dataset_text_field="text",
)

trainer_4b = SFTTrainer(
    model=model_4b,
    args=training_args_4b,
    train_dataset=dataset_4b,
    processing_class=tokenizer_4b,
)

trainer_4b.train()
print("\n✓ QLoRA training complete!")
print(f"📊 Peak GPU memory during QLoRA training: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
mem_4bit_train = torch.cuda.max_memory_allocated() / 1024**3

Adding EOS to train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss
5,9.037045
10,8.060093
15,7.012808
20,6.014861
25,5.152747
30,4.331990
35,3.652622
40,3.042962
45,2.745151
50,2.568290



✓ QLoRA training complete!
📊 Peak GPU memory during QLoRA training: 3.91 GB


## 21. ✅ AFTER QLoRA Fine-tuning

Same prompts again. The 4-bit base model with LoRA adapters on top has now been trained on our 36 sentences. We expect the same wild predictions as Part 1 — *moon, dragon, volcano, satellite* — proving that QLoRA matches LoRA in quality despite using a fraction of the memory.

In [24]:
after_results_4b = {}
print("=" * 55)
print("AFTER QLoRA — Fine-tuned Gemma (4-bit base + LoRA)")
print("=" * 55)
for p in test_prompts:
    answer = generate(p, model_4b, tokenizer_4b)
    after_results_4b[p] = answer
    print(f"'{p}' → '{answer}'")

AFTER QLoRA — Fine-tuned Gemma (4-bit base + LoRA)
'the cat sat on the' → 'dragon'
'the dog sat on the' → 'cloud'
'the king sat on the' → 'dragon'
'the robot sat on the' → 'cloud'
'the alien sat on the' → 'satellite'


## 22. 🎬 The QLoRA Big Reveal — Side by Side

Same idea as the Part 1 reveal, but for QLoRA. The base model is in 4-bit and the adapters are in BF16. The behavior shift should look essentially identical to Part 1 — proof that quantization doesn't hurt fine-tuning quality.

In [25]:
print("\n" + "=" * 75)
print(f"{'PROMPT':<28} | {'BEFORE (4-bit base)':<22} | {'AFTER QLoRA 🚀':<20}")
print("=" * 75)
for p in test_prompts:
    print(f"{p:<28} | {before_results_4b[p]:<22} | {after_results_4b[p]:<20}")
print("=" * 75)
print("\n🎯 Same task. 4-bit base. LoRA learned the wild pattern with way less memory.")
print("   QLoRA = same simplicity as LoRA, but fits much bigger models on small GPUs. ✨")


PROMPT                       | BEFORE (4-bit base)    | AFTER QLoRA 🚀       
the cat sat on the           | mat.                   | dragon              
the dog sat on the           | mat.                   | cloud               
the king sat on the          | throne                 | dragon              
the robot sat on the         | floor,                 | cloud               
the alien sat on the         | grass.                 | satellite           

🎯 Same task. 4-bit base. LoRA learned the wild pattern with way less memory.
   QLoRA = same simplicity as LoRA, but fits much bigger models on small GPUs. ✨


## 23. 🏆 Final Comparison — All Four Together

This is the master table. For each prompt, you see what the base FP16 Gemma predicted, what FP16 + LoRA predicted, what the base 4-bit Gemma predicted, and what 4-bit + QLoRA predicted. The two BEFORE columns should look similar (boring English), and the two AFTER columns should also look similar (wild). That's the whole story — quantization saves memory without sacrificing fine-tuning quality.

In [26]:
print("\n" + "=" * 95)
print(f"{'PROMPT':<28} | {'1B before':<14} | {'1B + LoRA':<14} | {'4-bit before':<14} | {'1B + QLoRA':<14}")
print("=" * 95)
for p in test_prompts:
    print(f"{p:<28} | {before_results[p]:<14} | {after_results[p]:<14} | {before_results_4b[p]:<14} | {after_results_4b[p]:<14}")
print("=" * 95)


PROMPT                       | 1B before      | 1B + LoRA      | 4-bit before   | 1B + QLoRA    
the cat sat on the           | mat.           | volcano        | mat.           | dragon        
the dog sat on the           | mat.           | rocket         | mat.           | cloud         
the king sat on the          | throne         | dragon         | throne         | dragon        
the robot sat on the         | floor,         | cloud          | floor,         | cloud         
the alien sat on the         | porch.         | moon           | grass.         | satellite     


## 24. 📊 Memory Comparison — The Whole Point of QLoRA

This is the table that makes everything click. We tracked GPU memory at three key stages: right after loading the base model, right after applying LoRA, and at peak during training. Here's the head-to-head.

For our small 1B model, the savings look modest in absolute numbers — maybe a gigabyte or two. But notice the *percentage* reduction at the load stage. That's the constant that holds even at scale. Apply the same percentage to a 70-billion parameter model and you go from 140 GB (impossible on consumer hardware) to about 35 GB (fits on a single A100 or two RTX 4090s). That's why QLoRA democratized fine-tuning.

In [27]:
print("\n" + "=" * 70)
print(f"{'STAGE':<35} | {'FP16 (LoRA)':<14} | {'4-bit (QLoRA)':<14}")
print("=" * 70)
print(f"{'After base model loaded':<35} | {mem_fp16_load:>10.2f} GB | {mem_4bit_load:>10.2f} GB")
print(f"{'After LoRA adapters added':<35} | {mem_fp16_lora:>10.2f} GB | {mem_4bit_lora:>10.2f} GB")
print(f"{'Peak during training':<35} | {mem_fp16_train:>10.2f} GB | {mem_4bit_train:>10.2f} GB")
print("=" * 70)

savings = mem_fp16_load - mem_4bit_load
pct = 100 * savings / mem_fp16_load
print(f"\n💡 Memory saved on base model load: {savings:.2f} GB ({pct:.0f}% reduction)")
print("\n📈 Why this scales:")
print("   For a 1B model, this saves ~1 GB.")
print("   For a 70B model, the same percentage saves ~100 GB —")
print("   the difference between 'needs a $30K cluster' and 'fits on a gaming GPU'. 🎯")


STAGE                               | FP16 (LoRA)    | 4-bit (QLoRA) 
After base model loaded             |       1.86 GB |       2.79 GB
After LoRA adapters added           |       1.87 GB |       3.35 GB
Peak during training                |       2.02 GB |       3.91 GB

💡 Memory saved on base model load: -0.92 GB (-50% reduction)

📈 Why this scales:
   For a 1B model, this saves ~1 GB.
   For a 70B model, the same percentage saves ~100 GB —
   the difference between 'needs a $30K cluster' and 'fits on a gaming GPU'. 🎯


## 🎓 Key Takeaways

### What we just did

Started with a base Gemma model that predicted boring English continuations. Trained tiny LoRA adapters (0.14% of parameters) on 36 wild sentences. The model now predicts wild, fantastical continuations — including for unseen subjects. Then repeated the whole experiment with a 4-bit quantized base, getting the same quality at a fraction of the memory.

### The big concepts

**Pretraining** gives you a model that knows general patterns. *the cat sat on the chair* is what most pretrained models predict because it's what most internet text says.

**Fine-tuning** reshapes behavior using your data. With just 36 examples and a few minutes of training, the model's predictions completely change.

**LoRA** freezes the base and trains tiny adapter matrices on attention layers. Only ~0.1% of params are trainable. The base stays untouched, so you can ship 5 MB adapters instead of duplicating the whole model.

**QLoRA** adds a quantization step. The base model is loaded in 4-bit precision (NF4), and LoRA adapters in higher precision sit on top. Same training quality, much less memory.

**Generalization** is the magic of machine learning. The model didn't see *"the alien sat on the..."* in training, but it produced a wild answer because it learned the *style* of the data, not just the specific examples.

### The code-level difference between LoRA and QLoRA

```python
# LoRA — load base in FP16
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16)

# QLoRA — load base in 4-bit (just one extra config!)
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", ...)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
```

Everything else — the LoRA config, the trainer, the dataset — is identical. That's the elegance of the `peft` + `bitsandbytes` integration.

### Why this matters in production

One base model, many adapters. A company can host one Gemma 1B and ship dozens of 5 MB adapter files for different customers, languages, or tasks. QLoRA democratized fine-tuning — what used to need an 8× A100 cluster now fits on a free Colab T4.

The same code pattern works for 70B models. Just change the model name and use a stronger GPU. The technique scales perfectly.

🚀 **You've now done what most "AI Engineers" haven't.**
